# Bi-Encoder Interactive Walkthrough
Run every cell top to bottom. Each cell isolates one concept from `src/model.py`.
You will see the actual tensor shapes and values at each step.

In [ ]:
import torch
import torch.nn.functional as F
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

---
## Part 1 — Simulating Tura's Output
Before touching the model, let's build fake tensors that look exactly like what
Tura's DataLoader will give you. This makes the shapes concrete.

In [ ]:
BATCH_SIZE = 2
MAX_LENGTH = 10   # Short for readability — real value is 256
VOCAB_SIZE = 30522  # DistilBERT vocabulary size

# Fake token IDs — in reality these are the output of the tokenizer
input_ids = torch.randint(low=100, high=5000, size=(BATCH_SIZE, MAX_LENGTH))

# Sample 0: 7 real tokens, 3 padding
# Sample 1: 4 real tokens, 6 padding
attention_mask = torch.tensor([
    [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],  # 7 real tokens
    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],  # 4 real tokens
])

print('input_ids shape:', input_ids.shape)        # [2, 10]
print('attention_mask shape:', attention_mask.shape)  # [2, 10]
print()
print('input_ids:')
print(input_ids)
print()
print('attention_mask (1=real, 0=padding):')
print(attention_mask)

---
## Part 2 — Mean Pooling Step by Step

We'll use **fake token embeddings** (random numbers) instead of running DistilBERT,
so you can see exactly what mean_pooling does to the numbers.

In [ ]:
HIDDEN_SIZE = 4  # Tiny hidden size for readability — real DistilBERT uses 768

# Pretend this is last_hidden_state from DistilBERT
torch.manual_seed(42)
token_embeddings = torch.randn(BATCH_SIZE, MAX_LENGTH, HIDDEN_SIZE)

print('token_embeddings shape:', token_embeddings.shape)  # [2, 10, 4]
print('Sample 0, all tokens (first 4 dims):')
print(token_embeddings[0].round(decimals=2))
print('\nNote: positions 7,8,9 are PADDING — we must ignore them!')

In [ ]:
# ----- STEP 1: Expand the mask -----
# attention_mask: [2, 10] -> expand to [2, 10, 4]

expanded_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

print('After .unsqueeze(-1):', attention_mask.unsqueeze(-1).shape)  # [2, 10, 1]
print('After .expand(...)  :', expanded_mask.shape)                 # [2, 10, 4]
print()
print('expanded_mask[0] (sample 0):')
print(expanded_mask[0])  # Rows 7,8,9 should be all zeros

In [ ]:
# ----- STEP 2: Zero out padding and sum -----

masked_embeddings = token_embeddings * expanded_mask

print('token_embeddings[0, 6] (last real token):', token_embeddings[0, 6].round(decimals=2))
print('masked_embeddings[0, 6] (should be same):', masked_embeddings[0, 6].round(decimals=2))
print()
print('token_embeddings[0, 7] (first padding):', token_embeddings[0, 7].round(decimals=2))
print('masked_embeddings[0, 7] (should be zeros):', masked_embeddings[0, 7].round(decimals=2))

sum_embeddings = masked_embeddings.sum(dim=1)
print('\nsum_embeddings shape:', sum_embeddings.shape)  # [2, 4]

In [ ]:
# ----- STEP 3: Count real tokens and divide -----

token_counts = expanded_mask.sum(dim=1).clamp(min=1e-9)
print('token_counts shape:', token_counts.shape)  # [2, 4]
print('token_counts:')
print(token_counts)
print()
print('Sample 0 has 7 real tokens -> each column count = 7.0')
print('Sample 1 has 4 real tokens -> each column count = 4.0')

pooled = sum_embeddings / token_counts
print('\npooled shape:', pooled.shape)  # [2, 4]
print('pooled (the sentence embeddings before normalization):')
print(pooled.round(decimals=3))

In [ ]:
# VERIFY: Let's manually compute sample 0 to double-check
manual_sum = token_embeddings[0, :7, :].sum(dim=0)  # Sum of 7 real tokens
manual_mean = manual_sum / 7.0

print('Manual mean (sample 0):', manual_mean.round(decimals=3))
print('Pooled mean (sample 0):', pooled[0].round(decimals=3))
print('Match:', torch.allclose(manual_mean, pooled[0], atol=1e-5))

---
## Part 3 — L2 Normalization
This step projects every embedding onto the unit hypersphere.

In [ ]:
# Before normalization
norms_before = pooled.norm(p=2, dim=1)
print('L2 norms BEFORE normalization:', norms_before.round(decimals=3))
print('(These are arbitrary — depend on the token embeddings and pooling)')

normalized = F.normalize(pooled, p=2, dim=1)

norms_after = normalized.norm(p=2, dim=1)
print('\nL2 norms AFTER normalization:', norms_after.round(decimals=3))
print('(Should be exactly 1.0 for both samples)')

In [ ]:
# Demonstrate: after normalization, cosine similarity == dot product
emb_a = normalized[0]
emb_b = normalized[1]

cosine_sim = F.cosine_similarity(emb_a.unsqueeze(0), emb_b.unsqueeze(0))
dot_product = (emb_a * emb_b).sum()

print('Cosine similarity:', cosine_sim.item())
print('Dot product:       ', dot_product.item())
print('Are they equal?', torch.allclose(cosine_sim, dot_product, atol=1e-6))
print()
print('This is WHY we normalize — cosine sim becomes a free dot product.')

---
## Part 4 — Running the Full BiEncoder
Now we run the actual model from `src/model.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.model import BiEncoder, mean_pooling

# This will download DistilBERT (~250MB) on first run
model = BiEncoder()
model.eval()
print('Model loaded. Hidden size:', model.hidden_size)
print('Total parameters:', sum(p.numel() for p in model.parameters()):,)

In [ ]:
# Use real-ish shapes matching the actual contract with Tura
REAL_BATCH = 2
REAL_MAX_LEN = 16  # 256 in production, 16 here for speed

ids = torch.randint(0, 30522, (REAL_BATCH, REAL_MAX_LEN))
mask = torch.ones(REAL_BATCH, REAL_MAX_LEN, dtype=torch.long)
mask[0, -6:] = 0   # Simulate padding on sample 0

with torch.no_grad():
    embeddings = model(ids, mask)

print('Output shape:', embeddings.shape)          # Must be [2, 768]
print('L2 norms:', embeddings.norm(p=2, dim=1))   # Must be [1.0, 1.0]

assert embeddings.shape == (REAL_BATCH, 768), 'Shape is wrong!'
assert torch.allclose(embeddings.norm(p=2, dim=1), torch.ones(REAL_BATCH), atol=1e-5)
print('\nAll assertions passed — your module is working correctly.')

---
## Part 5 — Simulating the Full Triplet Pipeline
This is what Week 4 training will look like end-to-end (without Ana's actual loss function).

In [ ]:
def fake_triplet_loss(anchor, positive, negative, margin=0.3):
    """Placeholder for Ana's actual loss. Uses cosine distance."""
    # Since embeddings are L2-normalized, dot product == cosine similarity
    pos_sim = (anchor * positive).sum(dim=1)  # [B]
    neg_sim = (anchor * negative).sum(dim=1)  # [B]
    # Triplet loss: max(0, margin - (pos_sim - neg_sim))
    loss = torch.clamp(margin - (pos_sim - neg_sim), min=0.0)
    return loss.mean()

# Simulate one training batch (Tura provides these)
q_ids   = torch.randint(0, 30522, (4, 16))  # batch_size=4, max_len=16
q_mask  = torch.ones(4, 16, dtype=torch.long)
p_ids   = torch.randint(0, 30522, (4, 16))
p_mask  = torch.ones(4, 16, dtype=torch.long)
n_ids   = torch.randint(0, 30522, (4, 16))
n_mask  = torch.ones(4, 16, dtype=torch.long)

# Your module runs three times on the batch
with torch.no_grad():
    q_emb = model(q_ids, q_mask)
    p_emb = model(p_ids, p_mask)
    n_emb = model(n_ids, n_mask)

loss = fake_triplet_loss(q_emb, p_emb, n_emb)

print('q_emb shape:', q_emb.shape)   # [4, 768]
print('p_emb shape:', p_emb.shape)   # [4, 768]
print('n_emb shape:', n_emb.shape)   # [4, 768]
print('loss value: ', loss)          # scalar tensor
print('loss is scalar:', loss.shape == torch.Size([]))  # True
print()
print('This is exactly what Ana\'s loss function will receive from you.')

---
## Part 6 — What Happens if You Forget to Mask?

This cell shows **concretely** why masking matters.

In [ ]:
# Create a heavily padded input (only 2 real tokens out of 16)
sparse_ids = torch.zeros(1, 16, dtype=torch.long)
sparse_ids[0, 0] = 7592   # Token ID for 'hello'
sparse_ids[0, 1] = 2088   # Token ID for 'world'
# Rest are [PAD] tokens (id=0)

sparse_mask = torch.zeros(1, 16, dtype=torch.long)
sparse_mask[0, 0] = 1
sparse_mask[0, 1] = 1

with torch.no_grad():
    outputs = model.backbone(input_ids=sparse_ids, attention_mask=sparse_mask)
    hidden = outputs.last_hidden_state  # [1, 16, 768]

# CORRECT: masked mean pooling
correct_emb = mean_pooling(hidden, sparse_mask)   # averages only 2 tokens

# WRONG: naive averaging (averages all 16 including 14 padding tokens)
wrong_emb = hidden.mean(dim=1)

# How different are they?
diff = (correct_emb - wrong_emb).norm().item()
print(f'Difference between correct and naive pooling: {diff:.4f}')
print(f'(With 14/16 = 87.5% padding, the naive embedding is heavily corrupted)')

cos_sim = F.cosine_similarity(correct_emb, wrong_emb).item()
print(f'Cosine similarity between correct vs naive: {cos_sim:.4f}')
print(f'(1.0 = identical, lower = more corrupted by padding)')